In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd '/content/drive/MyDrive/Colab Notebooks/canAI2026'

/content/drive/MyDrive/Colab Notebooks/canAI2026


# Dependencies

In [3]:
# ============================================================================
# SECTION 1: INSTALL DEPENDENCIES
# ============================================================================

# !pip install aif360
!pip install fairlearn
!pip install alibi-detect
!pip install adversarial-robustness-toolbox
!pip install rtdl-revisiting-models
# !pip install xgboost


  Using cached adversarial_robustness_toolbox-1.20.1-py3-none-any.whl.metadata (10 kB)
Using cached adversarial_robustness_toolbox-1.20.1-py3-none-any.whl (1.1 MB)


In [4]:
# ============================================================================
# SECTION 2: IMPORTS
# ============================================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import shap
from typing import Optional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from rtdl_revisiting_models import FTTransformer

import os
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import joblib
import torch.optim as optim

from art.estimators.classification import XGBoostClassifier as ARTXGBoostClassifier
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import HopSkipJump
from alibi_detect.cd import MMDDrift

from art.attacks.inference.model_inversion import MIFace
# from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.attacks.inference.membership_inference import (
    MembershipInferenceBlackBox,
    ShadowModels
)
from sklearn.linear_model import LogisticRegression
# from privacy_meter.audit import Audit

# Data

## German

In [ ]:
from aif360.sklearn.datasets import fetch_german

X, y = fetch_german()

X.reset_index(drop=True, inplace=True)
y.reset_index(drop=True, inplace=True)

# Binarize labels
y = y.map({'good': 1, 'bad': 0})

y = y.to_numpy(dtype=int)

# Add binary_age column based on the condition, value = aged if age >= 25 else young
X['age_group'] = X['age'].apply(lambda x: 'aged' if x >= 25 else 'young')

In [ ]:

# Label encoding for categorical columns
X_enc = X.copy()
cat_cols = X_enc.select_dtypes(['object', 'category']).columns.tolist()
num_cols = [c for c in X_enc.columns if c not in cat_cols]

cardinalities = []
label_mappings = {}

for col in cat_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col])
    cardinalities.append(len(le.classes_))
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("\n=== Label Encoding Mappings ===")
for col, mapping in label_mappings.items():
    print(f"\n'{col}':")
    for original, encoded in mapping.items():
        print(f"  {original} → {encoded}")

print(f"\nCardinalities: {cardinalities}")


## Census Income

In [ ]:
# ============================================================================
# SECTION 3: DATA LOADING AND PREPROCESSING
# ============================================================================

# Load data
X, y = shap.datasets.adult(display=True)

# Label encoding for categorical columns
X_enc = X.copy()
cat_cols = X_enc.select_dtypes(['object', 'category']).columns.tolist()
num_cols = [c for c in X_enc.columns if c not in cat_cols]

cardinalities = []
label_mappings = {}

for col in cat_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col])
    cardinalities.append(len(le.classes_))
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("\n=== Label Encoding Mappings ===")
for col, mapping in label_mappings.items():
    print(f"\n'{col}':")
    for original, encoded in mapping.items():
        print(f"  {original} → {encoded}")

print(f"\nCardinalities: {cardinalities}")



=== Label Encoding Mappings ===

'Workclass':
   ? → 0
   Federal-gov → 1
   Local-gov → 2
   Never-worked → 3
   Private → 4
   Self-emp-inc → 5
   Self-emp-not-inc → 6
   State-gov → 7
   Without-pay → 8

'Marital Status':
   Divorced → 0
   Married-AF-spouse → 1
   Married-civ-spouse → 2
   Married-spouse-absent → 3
   Never-married → 4
   Separated → 5
   Widowed → 6

'Occupation':
   ? → 0
   Adm-clerical → 1
   Armed-Forces → 2
   Craft-repair → 3
   Exec-managerial → 4
   Farming-fishing → 5
   Handlers-cleaners → 6
   Machine-op-inspct → 7
   Other-service → 8
   Priv-house-serv → 9
   Prof-specialty → 10
   Protective-serv → 11
   Sales → 12
   Tech-support → 13
   Transport-moving → 14

'Relationship':
   Husband → 0
   Not-in-family → 1
   Other-relative → 2
   Own-child → 3
   Unmarried → 4
   Wife → 5

'Race':
   Amer-Indian-Eskimo → 0
   Asian-Pac-Islander → 1
   Black → 2
   Other → 3
   White → 4

'Sex':
   Female → 0
   Male → 1

'Country':
   ? → 0
   Cambodia → 1
  

## Diabetes

In [5]:
# ---------------------------
# Data Loading and Preparation
# ---------------------------
from fairlearn.datasets import fetch_diabetes_hospital

# Data processing
X, y = fetch_diabetes_hospital(as_frame=True, return_X_y=True)
# The columns readmit_binary and readmitted are included in X -> remove both to avoid target leakage
X.drop(columns=["readmitted", "readmit_binary"], inplace=True)

# Get a boolean mask for rows where gender is not 'Unknown/Invalid' (which was encoded as 2)
valid_gender_mask = X['gender'] != 'Unknown/Invalid'

# Filter both X and y using the same mask
X = X[valid_gender_mask]
y = y[valid_gender_mask]


In [6]:

# Label encoding for categorical columns
X_enc = X.copy()
cat_cols = X_enc.select_dtypes(['object', 'category']).columns.tolist()
num_cols = [c for c in X_enc.columns if c not in cat_cols]

cardinalities = []
label_mappings = {}

for col in cat_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col])
    cardinalities.append(len(le.classes_))
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("\n=== Label Encoding Mappings ===")
for col, mapping in label_mappings.items():
    print(f"\n'{col}':")
    for original, encoded in mapping.items():
        print(f"  {original} → {encoded}")

print(f"\nCardinalities: {cardinalities}")



=== Label Encoding Mappings ===

'race':
  AfricanAmerican → 0
  Asian → 1
  Caucasian → 2
  Hispanic → 3
  Other → 4
  Unknown → 5

'gender':
  Female → 0
  Male → 1

'age':
  '30 years or younger' → 0
  '30-60 years' → 1
  'Over 60 years' → 2

'discharge_disposition_id':
  'Discharged to Home' → 0
  Other → 1

'admission_source_id':
  Emergency → 0
  Other → 1
  Referral → 2

'medical_specialty':
  Cardiology → 0
  Emergency/Trauma → 1
  Family/GeneralPractice → 2
  InternalMedicine → 3
  Missing → 4
  Other → 5

'primary_diagnosis':
  'Genitourinary Issues' → 0
  'Musculoskeletal Issues' → 1
  'Respiratory Issues' → 2
  Diabetes → 3
  Other → 4

'max_glu_serum':
  >200 → 0
  >300 → 1
  None → 2
  Norm → 3

'A1Cresult':
  >7 → 0
  >8 → 1
  None → 2
  Norm → 3

'insulin':
  Down → 0
  No → 1
  Steady → 2
  Up → 3

'change':
  Ch → 0
  No → 1

'diabetesMed':
  No → 0
  Yes → 1

'medicare':
  False → 0
  True → 1

'medicaid':
  False → 0
  True → 1

'had_emergency':
  False → 0
  True 

In [25]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(sampling_strategy=0.2, random_state=7)
X_smote, y_smote = smote.fit_resample(X_enc, y)


## Split and Preprocess

In [27]:
# Split data
random_state = 7
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_smote, y_smote, test_size=0.2, random_state=random_state, stratify=y_smote
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=random_state, stratify=y_trainval
)

print(f"\n=== Data Shapes ===")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


=== Data Shapes ===
Train: (69431, 22), Val: (17358, 22), Test: (21698, 22)


In [28]:
# Convert to numpy arrays
X_train_np = X_train.values.astype(np.float32)
X_val_np = X_val.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_train_np = (y_train.values if isinstance(y_train, pd.Series) else y_train).astype(np.int64)
y_val_np = (y_val.values if isinstance(y_val, pd.Series) else y_val).astype(np.int64)
y_test_np = (y_test.values if isinstance(y_test, pd.Series) else y_test).astype(np.int64)

# For FT-Transformer: separate numerical and categorical
X_train_num = X_train[num_cols].values.astype(np.float32)
X_val_num = X_val[num_cols].values.astype(np.float32)
X_test_num = X_test[num_cols].values.astype(np.float32)

X_train_cat = X_train[cat_cols].values.astype(np.int64)
X_val_cat = X_val[cat_cols].values.astype(np.int64)
X_test_cat = X_test[cat_cols].values.astype(np.int64)

# For FT-Transformer ART wrapper: concatenate num + cat
X_train_art = np.concatenate([X_train_num, X_train_cat], axis=1).astype(np.float32)
X_test_art = np.concatenate([X_test_num, X_test_cat], axis=1).astype(np.float32)

# Clip values for adversarial attacks
clip_values = (
    X_test_np.min(axis=0),
    X_test_np.max(axis=0)
)
clip_values_global = (float(X_test_art.min()), float(X_test_art.max()))

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model

## Define

In [33]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        # Initialize weights uniformly
        nn.init.uniform_(self.fc1.weight, a=-1.0 / (input_dim**0.5), b=+1.0 / (input_dim**0.5))
        nn.init.zeros_(self.fc1.bias)
        nn.init.uniform_(self.fc2.weight, a=-1.0 / (hidden_dim**0.5), b=+1.0 / (hidden_dim**0.5))
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # returns logits

In [34]:
# ============================================================================
# SECTION 4: MODEL ARCHITECTURES
# ============================================================================

# --- TabResNet Architecture ---
class TabResBlock(nn.Module):
    def __init__(self, d_block, d_hidden, dropout1, dropout2):
        super(TabResBlock, self).__init__()
        self.bn1 = nn.BatchNorm1d(d_block)
        self.ln1 = nn.Linear(d_block, d_hidden)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout1)
        self.ln2 = nn.Linear(d_hidden, d_block)
        self.drop2 = nn.Dropout(dropout2)

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.ln1(out)
        out = self.relu1(out)
        out = self.drop1(out)
        out = self.ln2(out)
        out = self.drop2(out)
        out += identity
        return out

class TabResNet(nn.Module):
    def __init__(
        self,
        d_in: int,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        d_hidden: Optional[int],
        d_hidden_multiplier: Optional[float] = 2,
        dropout1: float = 0.2,
        dropout2: float = 0
    ):
        super(TabResNet, self).__init__()
        self.input_projection = nn.Linear(d_in, d_block)
        self.resblocks = nn.ModuleList([
            TabResBlock(d_block, d_hidden, dropout1, dropout2) for _ in range(n_blocks)
        ])
        self.predblock = (
            nn.Sequential(
                nn.BatchNorm1d(d_block),
                nn.ReLU(),
                nn.Linear(d_block, d_out)
            ) if d_out is not None else None
        )

    def forward(self, x):
        x = self.input_projection(x)
        for block in self.resblocks:
            x = block(x)
        if self.predblock is not None:
            x = self.predblock(x)
        return x


# --- FT-Transformer Wrapper for ART ---
class FTTransformerARTWrapper(nn.Module):
    def __init__(self, ft_model, n_num_features, cat_cardinalities):
        super().__init__()
        self.ft_model = ft_model
        self.n_num_features = n_num_features
        self.cat_cardinalities = cat_cardinalities

    def forward(self, x):
        x_num = x[:, :self.n_num_features]
        x_cat = x[:, self.n_num_features:]

        # Project categorical features safely for embeddings
        x_cat_proj = []
        for i, card in enumerate(self.cat_cardinalities):
            xi = x_cat[:, i]
            xi = torch.round(xi)
            xi = torch.clamp(xi, 0, card - 1)
            x_cat_proj.append(xi)

        x_cat_proj = torch.stack(x_cat_proj, dim=1).long()
        return self.ft_model(x_num, x_cat_proj)

## Train MLP, DT, SVM

In [35]:
# ============================================================================
# ADDITIONAL SECTION: TRAIN AND SAVE NEW MODELS (MLP, SVM, DECISION TREE)
# ============================================================================




print("\n" + "="*70)
print("TRAINING NEW MODELS")
print("="*70)

# --- 1. MLP Model ---
print("\n[1/3] Training MLP...")

torch.manual_seed(7)
np.random.seed(7)


# Instantiate MLP
input_dim = X_train_np.shape[1]
mlp_model = MLP(input_dim=input_dim, hidden_dim=50, output_dim=2).to(device)

# Training setup
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# Convert to tensors
X_train_mlp_t = torch.from_numpy(X_train_np).to(device)
y_train_mlp_t = torch.from_numpy(y_train_np).to(device)
X_val_mlp_t = torch.from_numpy(X_val_np).to(device)
y_val_mlp_t = torch.from_numpy(y_val_np).to(device)

# Training loop
epochs = 50
batch_size = 256
best_val_acc = 0

for epoch in range(epochs):
    mlp_model.train()

    # Mini-batch training
    indices = torch.randperm(len(X_train_mlp_t))
    for i in range(0, len(X_train_mlp_t), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch_X = X_train_mlp_t[batch_idx]
        batch_y = y_train_mlp_t[batch_idx]

        mlp_optimizer.zero_grad()
        outputs = mlp_model(batch_X)
        loss = mlp_criterion(outputs, batch_y)
        loss.backward()
        mlp_optimizer.step()

    # Validation
    if (epoch + 1) % 10 == 0:
        mlp_model.eval()
        with torch.no_grad():
            val_outputs = mlp_model(X_val_mlp_t)
            val_preds = torch.argmax(val_outputs, dim=1)
            val_acc = (val_preds == y_val_mlp_t).float().mean().item()

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(mlp_model.state_dict(), "saved_model/mlp_diabetes_2.pth")

            print(f"  Epoch {epoch+1}/{epochs} - Val Acc: {val_acc:.4f}")

print(f"  Best validation accuracy: {best_val_acc:.4f}")
print(f"  Model saved to: saved_model/mlp_diabetes_2.pth")


TRAINING NEW MODELS

[1/3] Training MLP...
  Epoch 10/50 - Val Acc: 0.8333
  Epoch 20/50 - Val Acc: 0.8332
  Epoch 30/50 - Val Acc: 0.8341
  Epoch 40/50 - Val Acc: 0.8375
  Epoch 50/50 - Val Acc: 0.8392
  Best validation accuracy: 0.8392
  Model saved to: saved_model/mlp_diabetes.pth


In [14]:
# --- 2. SVM Model ---
print("\n[2/3] Training SVM...")

svm_model = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,  # Crucial for probability predictions
    random_state=42
)

# Train SVM
svm_model.fit(X_train_np, y_train_np)

# Evaluate on validation set
svm_val_acc = accuracy_score(y_val_np, svm_model.predict(X_val_np))
print(f"  Validation accuracy: {svm_val_acc:.4f}")

# Save model
joblib.dump(svm_model, "saved_model/svm_diabetes.pkl")
print(f"  Model saved to: saved_model/svm_diabetes.pkl")




[2/3] Training SVM...
  Validation accuracy: 0.8884
  Model saved to: saved_model/svm_diabetes.pkl


In [15]:
# --- 3. Decision Tree Model ---
print("\n[3/3] Training Decision Tree...")

dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=7
)

# Train Decision Tree
dt_model.fit(X_train_np, y_train_np)

# Evaluate on validation set
dt_val_acc = accuracy_score(y_val_np, dt_model.predict(X_val_np))
print(f"  Validation accuracy: {dt_val_acc:.4f}")

# Save model
joblib.dump(dt_model, "saved_model/dt_diabetes.pkl")
print(f"  Model saved to: saved_model/dt_diabetes.pkl")

print("\n" + "="*70)
print("ALL MODELS TRAINED AND SAVED")
print("="*70)


[3/3] Training Decision Tree...
  Validation accuracy: 0.8883
  Model saved to: saved_model/dt_diabetes.pkl

ALL MODELS TRAINED AND SAVED


## Load Model

In [36]:
# ============================================================================
# ADDITIONAL SECTION: LOAD AND EVALUATE NEW MODELS
# ============================================================================

print("\n=== Loading and Evaluating New Models ===")

# --- Load MLP ---
input_dim = X_train_np.shape[1]
mlp_model_eval = MLP(input_dim=input_dim, hidden_dim=50, output_dim=2).to(device)
mlp_model_eval.load_state_dict(
    torch.load("saved_model/mlp_diabetes_2.pth", map_location=device)
)
mlp_model_eval.eval()

X_test_mlp_t = torch.from_numpy(X_test_np).to(device)
y_test_mlp_t = torch.from_numpy(y_test_np).to(device)

with torch.no_grad():
    mlp_logits = mlp_model_eval(X_test_mlp_t)
    mlp_preds = torch.argmax(mlp_logits, dim=1)
    mlp_accuracy = (mlp_preds == y_test_mlp_t).float().mean().item()
    print(f"MLP Test Accuracy: {mlp_accuracy:.4f}")

# --- Load SVM ---
svm_model_eval = joblib.load("saved_model/svm_diabetes.pkl")
svm_preds = svm_model_eval.predict(X_test_np)
svm_accuracy = accuracy_score(y_test_np, svm_preds)
print(f"SVM Test Accuracy: {svm_accuracy:.4f}")

# --- Load Decision Tree ---
dt_model_eval = joblib.load("saved_model/dt_diabetes.pkl")
dt_preds = dt_model_eval.predict(X_test_np)
dt_accuracy = accuracy_score(y_test_np, dt_preds)
print(f"Decision Tree Test Accuracy: {dt_accuracy:.4f}")


=== Loading and Evaluating New Models ===
MLP Test Accuracy: 0.8392
SVM Test Accuracy: 0.8333
Decision Tree Test Accuracy: 0.8334


In [12]:
# ============================================================================
# SECTION 5: LOAD TRAINED MODELS
# ============================================================================

print("\n=== Loading Models ===")

# --- XGBoost ---
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.01,
    random_state=random_state
)
xgb_model.load_model("saved_model/xgb_diabetes.json")

xgb_preds = xgb_model.predict(X_test_np)
xgb_accuracy = accuracy_score(y_test_np, xgb_preds)
print(f"XGBoost Test Accuracy: {xgb_accuracy:.4f}")



=== Loading Models ===
XGBoost Test Accuracy: 0.8884


In [13]:
# --- TabResNet ---
d_in = X_test_np.shape[1]
tab_model = TabResNet(
    d_in=d_in,
    d_out=2,
    n_blocks=2,
    d_block=16,
    d_hidden=32,
    dropout1=0.2,
    dropout2=0.05
).to(device)

tab_model.load_state_dict(
    torch.load("saved_model/tabresnet_diabetes.pth", map_location=device)
)
tab_model.eval()

X_test_t = torch.from_numpy(X_test_np).to(device)
y_test_t = torch.from_numpy(y_test_np).to(device)

with torch.no_grad():
    logits = tab_model(X_test_t)
    preds = torch.argmax(logits, dim=1)
    tab_accuracy = (preds == y_test_t).float().mean().item()
    print(f"TabResNet Test Accuracy: {tab_accuracy:.4f}")


TabResNet Test Accuracy: 0.8884


In [14]:
# --- FT-Transformer ---
n_cont_features = len(num_cols)
d_out = 2

default_kwargs = FTTransformer.get_default_kwargs()
default_kwargs['d_block'] = 32
default_kwargs['attention_n_heads'] = 1

ft_model = FTTransformer(
    n_cont_features=n_cont_features,
    cat_cardinalities=cardinalities,
    d_out=d_out,
    **default_kwargs,
).to(device)

ft_model.load_state_dict(
    torch.load("saved_model/fttransformer_diabetes.pth", map_location=device)
)
ft_model.eval()

X_test_num_t = torch.from_numpy(X_test_num).to(device)
X_test_cat_t = torch.from_numpy(X_test_cat).to(device)

with torch.no_grad():
    logits = ft_model(X_test_num_t, X_test_cat_t)
    preds = torch.argmax(logits, dim=1)
    ft_accuracy = (preds == y_test_t).float().mean().item()
    print(f"FT-Transformer Test Accuracy: {ft_accuracy:.4f}")

FT-Transformer Test Accuracy: 0.8884


# Robustness ----------

## HSJA

### Define wrapper

In [15]:
# ============================================================================
# SECTION 6: HSJA ATTACK SETUP
# ============================================================================

# ============================================================================
# ADDITIONAL SECTION: HSJA ATTACKS ON NEW MODELS
# ============================================================================

print("\n" + "="*70)
print("RUNNING HSJA ATTACKS ON NEW MODELS")
print("="*70)

# --- Update UnifiedModel to support sklearn models ---
class UnifiedModel:
    """Unifies predict() interface across all model types"""

    def __init__(self, model, model_type, device="cpu", n_cont_features=None):
        self.model = model
        self.model_type = model_type
        self.device = device
        self.n_cont_features = n_cont_features

    def predict_proba(self, X):
        if self.model_type in ["xgb", "svm", "dt"]:
            return self.model.predict_proba(X)

        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            if self.model_type in ["tabresnet", "mlp"]:
                logits = self.model(X_t)
            elif self.model_type == "ftt":
                num = X_t[:, :self.n_cont_features]
                cat = X_t[:, self.n_cont_features:].long()
                logits = self.model(num, cat)

            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        return probs

    def predict(self, X):
        if self.model_type in ["xgb", "svm", "dt"]:
            return self.model.predict(X)

        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)


# --- ART Classifier Builder for sklearn models ---
def build_art_sklearn(model, model_type, X_train, clip_vals):
    """Build ART classifier for sklearn models (SVM, Decision Tree)"""
    from art.estimators.classification import SklearnClassifier

    return SklearnClassifier(
        model=model,
        clip_values=clip_vals
    )

# --- ART Classifier Builders ---
def build_art_xgb(xgb_model, X_train, clip_vals):
    return ARTXGBoostClassifier(
        model=xgb_model,
        nb_features=X_train.shape[1],
        nb_classes=2,
        clip_values=clip_vals
    )

def build_art_pytorch(model, X_train, clip_vals):
    """Generic PyTorch classifier for ART"""
    return PyTorchClassifier(
        model=model,
        loss=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
        input_shape=(X_train.shape[1],),
        nb_classes=2,
        clip_values=clip_vals,
        device_type="gpu" if torch.cuda.is_available() else "cpu"
    )


RUNNING HSJA ATTACKS ON NEW MODELS


### Define run_hsja_attack()

In [42]:
# --- HSJA Attack Function ---
def run_hsja_attack(art_clf, model_wrapper, X_test, y_test, max_samples=500):
    """
    Run HopSkipJump Attack (decision-based adversarial attack)

    Args:
        art_clf: ART classifier wrapper
        model_wrapper: UnifiedModel instance for predictions
        X_test: Test features
        y_test: Test labels
        max_samples: Number of samples to attack (use fewer for speed)
    """
    # Subsample for computational efficiency
    if max_samples is not None and max_samples < len(X_test):
        idx = np.random.choice(len(X_test), max_samples, replace=False)
        X_attack = X_test[idx]
        y_attack = y_test[idx]
    else:
        X_attack = X_test
        y_attack = y_test

    print(f"  Running HSJA on {len(X_attack)} samples...")

    # Check wrapper gives correct predictions
    y_pred_wrapper = model_wrapper.predict(X_attack)
    acc_wrapper = (y_pred_wrapper == y_attack).mean()

    # Check ART wrapper matches
    y_pred_art = np.argmax(art_clf.predict(X_attack), axis=1)
    acc_art = (y_pred_art == y_attack).mean()

    # Alert if mismatch
    if abs(acc_wrapper - acc_art) > 0.01:
        print(f"⚠️  WARNING: Predictions differ by {abs(acc_wrapper - acc_art):.4f}")

    # Initialize attack
    attacker = HopSkipJump(
        classifier=art_clf,
        max_iter=10,
        init_eval=1000,
        init_size=1000
    )

    # Generate adversarial examples
    X_adv = attacker.generate(X_attack)

    # Evaluate
    y_clean = model_wrapper.predict(X_attack)
    y_adv = model_wrapper.predict(X_adv)

    acc_clean = (y_clean == y_attack).mean()
    acc_adv = (y_adv == y_attack).mean()
    acc_gap = acc_clean - acc_adv

    return {
        "acc_clean": acc_clean,
        "acc_adv": acc_adv,
        "acc_gap": acc_gap,
        "success_rate": (y_clean != y_adv).mean(),
        "hsja_robustness": 1 - acc_gap
    }

### Decision Tree

In [18]:
# --- Attack Decision Tree ---
print("\n[1/6] Decision Tree")
dt_wrapper = UnifiedModel(dt_model_eval, model_type="dt")
art_dt = build_art_sklearn(dt_model_eval, "dt", X_train_np, clip_values)


[1/6] Decision Tree


In [19]:


results_dt = run_hsja_attack(
    art_dt,
    dt_wrapper,
    X_test_np,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_dt['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_dt['acc_adv']:.4f}")
print(f"  Gap:       {results_dt['acc_gap']:.4f}")
print(f"  Success:   {results_dt['success_rate']:.4f}")

  Running HSJA on 500 samples...


HopSkipJump:   0%|          | 0/500 [00:00<?, ?it/s]

  Clean Acc: 0.8860
  Adv Acc:   0.1140
  Gap:       0.7720
  Success:   1.0000


### XGB

In [31]:
# --- Attack XGBoost ---
print("\n[2/6] XGBoost")
xgb_wrapper = UnifiedModel(xgb_model, model_type="xgb")
art_xgb = build_art_xgb(xgb_model, X_train_np, clip_values)


[2/6] XGBoost


In [32]:
# ============================================================================
# SECTION 7: RUN HSJA ATTACKS
# ============================================================================


results_xgb = run_hsja_attack(
    art_xgb,
    xgb_wrapper,
    X_test_np,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_xgb['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_xgb['acc_adv']:.4f}")
print(f"  Gap:       {results_xgb['acc_gap']:.4f}")
print(f"  Success:   {results_xgb['success_rate']:.4f}")

  Running HSJA on 500 samples...


HopSkipJump:   0%|          | 0/500 [00:00<?, ?it/s]

  Clean Acc: 0.8140
  Adv Acc:   0.8140
  Gap:       0.0000
  Success:   0.0000


### SVM

In [47]:
# --- Attack SVM ---
print("\n[3/6] SVM")
svm_wrapper = UnifiedModel(svm_model_eval, model_type="svm")
art_svm = build_art_sklearn(svm_model_eval, "svm", X_train_np, clip_values)


[3/6] SVM


In [62]:

results_svm = run_hsja_attack(
    art_svm,
    svm_wrapper,
    X_test_np,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_svm['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_svm['acc_adv']:.4f}")
print(f"  Gap:       {results_svm['acc_gap']:.4f}")
print(f"  Success:   {results_svm['success_rate']:.4f}")


  Running HSJA on 1000 samples...


HopSkipJump:   0%|          | 0/1000 [00:00<?, ?it/s]

  Clean Acc: 0.8810
  Adv Acc:   0.8810
  Gap:       0.0000
  Success:   0.0000


### MLP

In [40]:
# --- Attack MLP ---
print("\n[4/6] MLP")
mlp_wrapper = UnifiedModel(mlp_model_eval, model_type="mlp", device=device)
art_mlp = build_art_pytorch(mlp_model_eval, X_train_np, clip_values)


[4/6] MLP


In [43]:


results_mlp = run_hsja_attack(
    art_mlp,
    mlp_wrapper,
    X_test_np,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_mlp['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_mlp['acc_adv']:.4f}")
print(f"  Gap:       {results_mlp['acc_gap']:.4f}")
print(f"  Success:   {results_mlp['success_rate']:.4f}")

  Running HSJA on 500 samples...


HopSkipJump:   0%|          | 0/500 [00:00<?, ?it/s]

  Clean Acc: 0.8380
  Adv Acc:   0.2260
  Gap:       0.6120
  Success:   0.8880


### TabResNet

In [49]:
# --- Attack TabResNet ---
print("\n[5/6] TabResNet")
tab_wrapper = UnifiedModel(tab_model, model_type="tabresnet", device=device)
art_tab = build_art_pytorch(tab_model, X_train_np, clip_values)


[5/6] TabResNet


In [64]:


results_tab = run_hsja_attack(
    art_tab,
    tab_wrapper,
    X_test_np,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_tab['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_tab['acc_adv']:.4f}")
print(f"  Gap:       {results_tab['acc_gap']:.4f}")
print(f"  Success:   {results_tab['success_rate']:.4f}")

  Running HSJA on 500 samples...


HopSkipJump:   0%|          | 0/500 [00:00<?, ?it/s]

  Clean Acc: 0.9080
  Adv Acc:   0.9080
  Gap:       0.0000
  Success:   0.0000


### FT-Transformer

In [50]:
# --- Attack FT-Transformer ---
print("\n[6/6] FT-Transformer")
wrapped_ft_model = FTTransformerARTWrapper(
    ft_model=ft_model,
    n_num_features=X_train_num.shape[1],
    cat_cardinalities=cardinalities
).to(device)

ft_wrapper = UnifiedModel(
    wrapped_ft_model,
    model_type= "tabresnet",  # Use tabresnet type since wrapper handles the split
    device=device
)

art_ft = build_art_pytorch(wrapped_ft_model, X_train_art, clip_values)


[6/6] FT-Transformer


In [65]:


results_ft = run_hsja_attack(
    art_ft,
    ft_wrapper,
    X_test_art,
    y_test_np,
    max_samples=500
)

print(f"  Clean Acc: {results_ft['acc_clean']:.4f}")
print(f"  Adv Acc:   {results_ft['acc_adv']:.4f}")
print(f"  Gap:       {results_ft['acc_gap']:.4f}")
print(f"  Success:   {results_ft['success_rate']:.4f}")

  Running HSJA on 500 samples...


HopSkipJump:   0%|          | 0/500 [00:00<?, ?it/s]

  Clean Acc: 0.8900
  Adv Acc:   0.8900
  Gap:       0.0000
  Success:   0.0000


## Drift Robustness Score

### Define ()

In [51]:
def drift_robustness_score_with_p_soft(drift_result, alpha=0.05):
    """
    Drift robustness score in (0, 1).
    Accepts negative MMD distances.
    Higher = more robust.
    """

    d = drift_result["data"]["distance"]
    dt = drift_result["data"]["distance_threshold"]
    p = drift_result["data"]["p_val"]

    if dt <= 0:
        return 0.0

    # Signed normalized margin (can be > 1 or < 0)
    m = (dt - d) / dt

    # Smooth squashing to (0, 1)
    margin_score = 1.0 / (1.0 + np.exp(-m))

    # Confidence weighting
    p_factor = min(1.0, p / alpha)

    return float(margin_score * p_factor)


def compute_drift_robustness(model_wrapper, X_train, X_test, model_name, n_ref=2000, alpha=0.05):
    print(f"\n  Computing drift for {model_name}...")

    # Predict probabilities
    probs_train = model_wrapper.predict_proba(X_train)
    probs_test = model_wrapper.predict_proba(X_test)

    # Subsample reference
    np.random.seed(42)
    n_ref = min(n_ref, len(probs_train))
    idx = np.random.choice(len(probs_train), n_ref, replace=False)

    # Drift detector
    cd = MMDDrift(probs_train[idx], p_val=alpha)
    drift_result = cd.predict(probs_test)

    # Robustness score
    robustness = drift_robustness_score_with_p_soft(drift_result, alpha)

    return {
        "model": model_name,
        "is_drift": bool(drift_result["data"]["is_drift"]),
        "mmd_distance": float(drift_result["data"]["distance"]),
        "distance_threshold": float(drift_result["data"]["distance_threshold"]),
        "p_val": float(drift_result["data"]["p_val"]),
        "drift_robustness": robustness,
        "n_ref": n_ref
    }


### Run eval

In [52]:
print("\n" + "="*70)
print("SECTION 4: DISTRIBUTION SHIFT ROBUSTNESS (MMDDrift)")
print("="*70)

drift_results = []

models = [
    ("XGBoost", xgb_wrapper, X_train_np, X_test_np),
    ("SVM", svm_wrapper, X_train_np, X_test_np),
    ("Decision Tree", dt_wrapper, X_train_np, X_test_np),
    ("MLP", mlp_wrapper, X_train_np, X_test_np),
    ("TabResNet", tab_wrapper, X_train_np, X_test_np),
    ("FT-Transformer", ft_wrapper, X_train_art, X_test_art),
]

for name, wrapper, X_tr, X_te in models:
    res = compute_drift_robustness(wrapper, X_tr, X_te, name, n_ref=2000)
    drift_results.append(res)

    print(f"  MMD Distance:       {res['mmd_distance']:.6f}")
    print(f"  Distance Threshold: {res['distance_threshold']:.6f}")
    print(f"  p-value:            {res['p_val']:.4f}")
    print(f"  Drift Detected:     {res['is_drift']}")
    print(f"  Drift Robustness:   {res['drift_robustness']:.4f}")



SECTION 4: DISTRIBUTION SHIFT ROBUSTNESS (MMDDrift)

  Computing drift for XGBoost...
  MMD Distance:       -0.000163
  Distance Threshold: 0.000435
  p-value:            0.7200
  Drift Detected:     False
  Drift Robustness:   0.7980

  Computing drift for SVM...
  MMD Distance:       -0.000173
  Distance Threshold: 0.000429
  p-value:            0.6500
  Drift Detected:     False
  Drift Robustness:   0.8027

  Computing drift for Decision Tree...
  MMD Distance:       -0.000118
  Distance Threshold: 0.000450
  p-value:            0.6700
  Drift Detected:     False
  Drift Robustness:   0.7794

  Computing drift for MLP...
  MMD Distance:       -0.000244
  Distance Threshold: 0.000524
  p-value:            0.9300
  Drift Detected:     False
  Drift Robustness:   0.8123

  Computing drift for TabResNet...
  MMD Distance:       -0.000189
  Distance Threshold: 0.000453
  p-value:            0.7900
  Drift Detected:     False
  Drift Robustness:   0.8048

  Computing drift for FT-Transf

# Privacy

## MI Attack Accuracy

### Define

In [53]:
# ============================================================
# SECTION 9: BLACK-BOX MEMBERSHIP INFERENCE (OFFICIAL STYLE)
# ============================================================

from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
import numpy as np

print("\n" + "="*70)
print("BLACK-BOX MEMBERSHIP INFERENCE (OFFICIAL PROCEDURE)")
print("="*70)


def run_mi_blackbox_official(
    art_clf,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    attack_train_ratio=0.5
):
    """
    Implements official ART notebook procedure exactly:
    - Split data for attack training
    - Train attack model
    - Infer on remaining samples
    - Compute final attack accuracy
    """

    print("\n" + "-"*60)
    print(f"[Black-Box Membership Attack] {model_name}")
    print("-"*60)

    # ---------------------------------------------------------
    # 1️⃣ Split data for attack training
    # ---------------------------------------------------------

    print(np.mean(art_clf.predict(X_train[:100]), axis=0))
    print(np.mean(art_clf.predict(X_test[:100]), axis=0))


    attack_train_size = int(len(X_train) * attack_train_ratio)
    attack_test_size = int(len(X_test) * attack_train_ratio)

    # ---------------------------------------------------------
    # 2️⃣ Train attack model
    # ---------------------------------------------------------

    bb_attack = MembershipInferenceBlackBox(art_clf, attack_model_type='rf')

    bb_attack.fit(
        X_train[:attack_train_size], y_train[:attack_train_size],
        X_test[:attack_test_size], y_test[:attack_test_size]
    )

    # ---------------------------------------------------------
    # 3️⃣ Infer membership on remaining samples
    # ---------------------------------------------------------

    inferred_train = bb_attack.infer(
        X_train[attack_train_size:],
        y_train[attack_train_size:]
    )

    inferred_test = bb_attack.infer(
        X_test[attack_test_size:],
        y_test[attack_test_size:]
    )

    print("Unique inferred_train:", np.unique(inferred_train))
    print("Unique inferred_test:", np.unique(inferred_test))


    # ---------------------------------------------------------
    # 4️⃣ Compute attack accuracy (OFFICIAL FORMULA)
    # ---------------------------------------------------------

    # Members accuracy
    train_acc = np.sum(inferred_train) / len(inferred_train)

    # Non-members accuracy
    test_acc = 1 - (np.sum(inferred_test) / len(inferred_test))

    # Final weighted accuracy
    attack_acc = (
        train_acc * len(inferred_train) +
        test_acc * len(inferred_test)
    ) / (len(inferred_train) + len(inferred_test))

    privacy_score = 1 - 2 * abs(attack_acc - 0.5)


    print(f"Members Accuracy:     {train_acc:.6f}")
    print(f"Non-Members Accuracy: {test_acc:.6f}")
    print(f"Final Attack Accuracy:{attack_acc:.6f}")
    print(f"Privacy Score:        {privacy_score:.6f}")

    return float(attack_acc)



BLACK-BOX MEMBERSHIP INFERENCE (OFFICIAL PROCEDURE)


### Run

In [54]:
# ============================================================
# RUN ON SELECTED MODELS
# ============================================================

selected_models = [
    "XGBoost",
    "SVM",
    "Decision Tree",
    "MLP",
    "TabResNet",
    "FT-Transformer"
]
# selected_models = ["XGBoost"]

privacy_model_registry = {
    "XGBoost": ("XGBoost", art_xgb, X_train_np, y_train_np, X_test_np, y_test_np),
    "SVM": ("SVM", art_svm, X_train_np, y_train_np, X_test_np, y_test_np),
    "Decision Tree": ("Decision Tree", art_dt, X_train_np, y_train_np, X_test_np, y_test_np),
    "MLP": ("MLP", art_mlp, X_train_np, y_train_np, X_test_np, y_test_np),
    "TabResNet": ("TabResNet", art_tab, X_train_np, y_train_np, X_test_np, y_test_np),
    "FT-Transformer": ("FT-Transformer", art_ft, X_train_art, y_train_np, X_test_art, y_test_np),
}

In [55]:

attack_results = []

for model_name in selected_models:

    if model_name not in privacy_model_registry:
        continue

    name, art_clf, X_tr, y_tr, X_te, y_te = privacy_model_registry[model_name]

    attack_acc = run_mi_blackbox_official(
        art_clf,
        X_tr, y_tr,
        X_te, y_te,
        name,
        attack_train_ratio=0.5
    )

    attack_results.append({
        "model": name,
        "attack_accuracy": attack_acc
    })


# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n" + "="*70)
print("BLACK-BOX ATTACK SUMMARY")
print("="*70)
print(f"\n{'Model':<20} {'Attack Accuracy':<20}")
print("-"*50)

for r in attack_results:
    print(f"{r['model']:<20} {r['attack_accuracy']:<20.6f}")

print("="*70)


------------------------------------------------------------
[Black-Box Membership Attack] XGBoost
------------------------------------------------------------
[0.86800927 0.13199073]
[0.87220573 0.12779434]
Unique inferred_train: [0. 1.]
Unique inferred_test: [0. 1.]
Members Accuracy:     0.995517
Non-Members Accuracy: 0.003734
Final Attack Accuracy:0.759365
Privacy Score:        0.481271

------------------------------------------------------------
[Black-Box Membership Attack] SVM
------------------------------------------------------------
[0.88781589 0.11218411]
[0.88830651 0.11169349]
Unique inferred_train: [0. 1.]
Unique inferred_test: [0. 1.]
Members Accuracy:     0.771404
Non-Members Accuracy: 0.228456
Final Attack Accuracy:0.642123
Privacy Score:        0.715753

------------------------------------------------------------
[Black-Box Membership Attack] Decision Tree
------------------------------------------------------------
[0.89200296 0.10799704]
[0.9004358 0.0995642]
Uni

## SHAPr

### Define

In [56]:
# ============================================================
# SECTION: SHAPr PRIVACY METRIC (WITH AUTO SUBSAMPLING)
# ============================================================

from art.metrics import SHAPr
import numpy as np


def compute_shapr_privacy_score(
    art_clf,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    max_train_samples=5000,
    max_test_samples=2000,
    random_state=42
):
    """
    Computes SHAPr privacy metric.
    Automatically subsamples dataset if too large.

    Returns normalized privacy score in (0,1).
    """

    print("\n" + "-"*60)
    print(f"[SHAPr Privacy Metric] {model_name}")
    print("-"*60)

    np.random.seed(random_state)

    # --------------------------------------------------------
    # 1️⃣ Subsample if dataset too large
    # --------------------------------------------------------

    if len(X_train) > max_train_samples:
        idx_train = np.random.choice(
            len(X_train),
            max_train_samples,
            replace=False
        )
        X_train_sub = X_train[idx_train]
        y_train_sub = y_train[idx_train]
        print(f"Subsampled training set: {len(X_train_sub)} samples")
    else:
        X_train_sub = X_train
        y_train_sub = y_train

    if len(X_test) > max_test_samples:
        idx_test = np.random.choice(
            len(X_test),
            max_test_samples,
            replace=False
        )
        X_test_sub = X_test[idx_test]
        y_test_sub = y_test[idx_test]
        print(f"Subsampled test set: {len(X_test_sub)} samples")
    else:
        X_test_sub = X_test
        y_test_sub = y_test

    # --------------------------------------------------------
    # 2️⃣ Compute SHAPr leakage
    # --------------------------------------------------------

    shapr_leakage = SHAPr(
        art_clf,
        X_train_sub,
        y_train_sub,
        X_test_sub,
        y_test_sub
    )

    avg_leakage = float(np.mean(shapr_leakage))
    max_leakage = float(np.max(shapr_leakage))
    leak_rate = float(np.mean(shapr_leakage > 0.0))  # fraction of records with phi > 0


    # --------------------------------------------------------
    # 3️⃣ Normalize to Privacy Score (Higher = Better)
    # --------------------------------------------------------

    # SHAPr leakage is positive, unbounded
    # We convert leakage → privacy via inverse scaling

    privacy_score = 1 / (1 + avg_leakage)


    print(f"Average SHAPr Leakage: {avg_leakage:.4f}")
    print(f"Leak Rate (phi > 0):   {leak_rate:.4f}")
    print(f"SHAPr Privacy Score:   {privacy_score:.6f}")

    return privacy_score # float(np.clip(privacy_score, 0, 1))


### Run

In [57]:
print("\n" + "="*70)
print("RUNNING SHAPr ON SELECTED MODELS")
print("="*70)

shapr_results = []

for model_name in selected_models:

    name, art_clf, X_tr, y_tr, X_te, y_te = privacy_model_registry[model_name]

    shapr_score = compute_shapr_privacy_score(
        art_clf,
        X_tr,
        y_tr,
        X_te,
        y_te,
        name
    )

    shapr_results.append({
        "model": name,
        "shapr_privacy_score": shapr_score
    })



RUNNING SHAPr ON SELECTED MODELS

------------------------------------------------------------
[SHAPr Privacy Metric] XGBoost
------------------------------------------------------------
Subsampled training set: 5000 samples
Subsampled test set: 2000 samples
Average SHAPr Leakage: 0.8265
Leak Rate (phi > 0):   0.8892
SHAPr Privacy Score:   0.547495

------------------------------------------------------------
[SHAPr Privacy Metric] SVM
------------------------------------------------------------
Subsampled training set: 5000 samples
Subsampled test set: 2000 samples
Average SHAPr Leakage: 0.7970
Leak Rate (phi > 0):   0.8884
SHAPr Privacy Score:   0.556483

------------------------------------------------------------
[SHAPr Privacy Metric] Decision Tree
------------------------------------------------------------
Subsampled training set: 5000 samples
Subsampled test set: 2000 samples
Average SHAPr Leakage: 0.7965
Leak Rate (phi > 0):   0.8888
SHAPr Privacy Score:   0.556638

---------